# Opal walkthrough

Institutional option pricing from Python: vanilla and exotic equity options, stochastic vol, and interest rate derivatives.

Install first (from the repo root): `pip install ./python`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import opal

opal.__version__

## 1. Vanilla pricing and greeks

In [ ]:
g = opal.bs_greeks('call', spot=100, strike=105, expiry=0.5, rate=0.04, div=0.015, vol=0.22)
print(f'price {g.price:.4f}  delta {g.delta:.4f}  gamma {g.gamma:.4f}')
print(f'vega/pt {g.vega/100:.4f}  theta/day {g.theta/365:.4f}  rho/1% {g.rho/100:.4f}')

## 2. The volatility smile under Heston vs flat Black-Scholes

In [ ]:
hp = opal.HestonParams(v0=0.04, kappa=1.5, theta=0.05, xi=0.6, rho=-0.7)
strikes = np.arange(70, 131, 2.5)
ivs = []
for k in strikes:
    px = opal.heston_price('call', spot=100, strike=float(k), expiry=1.0, rate=0.03, div=0.0, params=hp)
    ivs.append(opal.implied_vol('call', price=px, spot=100, strike=float(k), expiry=1.0, rate=0.03))

plt.figure(figsize=(8, 4))
plt.plot(strikes, np.array(ivs) * 100, marker='o', ms=3, label='Heston-implied')
plt.axhline(20, ls='--', c='gray', label='flat 20%')
plt.xlabel('strike'); plt.ylabel('implied vol (%)'); plt.legend(); plt.title('Equity skew from Heston')
plt.show()

## 3. Exotics: barriers, Asians, lookbacks

In [ ]:
common = dict(spot=100, strike=100, expiry=1.0, rate=0.05, vol=0.25)
print('vanilla      ', round(opal.bs_price('call', **common), 4))
print('up-and-out KO', round(opal.barrier_price('call', 'up-out', barrier=125, **common), 4))
print('asian (arith)', round(opal.asian_price('call', **common), 4))
print('asian (geo)  ', round(opal.asian_price('call', average='geometric', **common), 4))
print('lookback flt ', round(opal.lookback_price('call', spot=100, expiry=1.0, rate=0.05, vol=0.25), 4))

# Monte Carlo cross-check with standard errors
mc = opal.mc_price('call', payoff='asian-arith', paths=100000, steps=252, **common)
print(f'asian MC      {mc.price:.4f} +/- {mc.std_error:.4f}')

## 4. Custom payoffs in pure Python

Any path-dependent European payoff can be priced through `mc_custom` - here a capped cliquet-style payoff.

In [ ]:
def capped_calls(path):
    # sum of monthly returns, each capped at 3%
    rets = np.diff(np.concatenate([[100.0], path])) / np.concatenate([[100.0], path[:-1]])
    return max(np.minimum(rets, 0.03).sum(), 0.0) * 100

res = opal.mc_custom(capped_calls, spot=100, expiry=1.0, rate=0.04, vol=0.2, paths=20000, steps=12)
print(f'cliquet price {res.price:.4f} +/- {res.std_error:.4f}')

## 5. Interest rates: caps, swaptions, SABR and Hull-White

In [ ]:
curve = opal.DiscountCurve([1.0, 2.0, 5.0, 10.0], [0.042, 0.040, 0.039, 0.041])

cap = opal.cap_floor_price(curve, strike=0.04, vol=0.3, first_fixing=0.25, maturity=5.0, tau=0.25)
print(f'5y cap @4%: {cap.price * 100:.4f} per 100 notional ({len(cap.caplets)} caplets)')

sw = opal.swaption_price(curve, 'payer', strike=0.04, vol=0.25, expiry=1.0, tenor=5.0)
print(f'1y5y payer swaption: {sw.price * 100:.4f}  (fwd swap {sw.forward_swap_rate:.4%}, annuity {sw.annuity:.4f})')

# SABR swaption smile
sp = opal.SabrParams(alpha=0.04, beta=0.5, rho=-0.25, nu=0.45)
ks = np.linspace(0.02, 0.06, 41)
vols = [opal.sabr_vol(sw.forward_swap_rate, float(k), 1.0, sp) for k in ks]
plt.figure(figsize=(8, 4))
plt.plot(ks * 100, np.array(vols) * 100)
plt.xlabel('strike (%)'); plt.ylabel('Black vol (%)'); plt.title('SABR swaption smile')
plt.show()

hw = opal.HullWhiteParams(a=0.08, sigma=0.011)
print(f'HW 1y caplet on 3m rate @4%: {opal.hw_caplet(curve, hw, 1.0, 1.25, 0.04) * 100:.4f}')

## 6. Position monitoring: spot/vol P&L grid

In [ ]:
qty = -100 * 100  # short 100 contracts
base = opal.bs_price('call', spot=100, strike=105, expiry=0.5, rate=0.04, vol=0.22)
spots = np.linspace(85, 115, 31)
fig, ax = plt.subplots(figsize=(8, 4))
for dv in (-5, 0, 5):
    pnl = [qty * (opal.bs_price('call', spot=float(s), strike=105, expiry=0.5, rate=0.04, vol=0.22 + dv / 100) - base) for s in spots]
    ax.plot(spots, pnl, label=f'vol {dv:+d}pt')
ax.axhline(0, c='gray', lw=0.5); ax.legend(); ax.set_xlabel('spot'); ax.set_ylabel('P&L')
ax.set_title('Short 100x 105C: P&L vs spot and vol')
plt.show()